# 03 — Discovery modes

When `xr.open_dataset(..., engine="edr")` is called, `edr-xarray` must
figure out the actual coordinate arrays for every axis (longitude,
latitude, vertical levels, time). The library offers **three discovery
strategies**, controlled by the `discovery` keyword:

| Mode | Network cost | Coord resolution | Behavior |
|---|---|---|---|
| `"probe"` *(default)* | +1 cube request | full | fetches a CoverageJSON to read its `domain.axes` |
| `"metadata_only"` | 0 | minimal | uses only the metadata `extent.spatial.bbox` and `extent.temporal.values` |
| `"strict"` | 0 | minimal | like `metadata_only` but raises if `extent.temporal.values` is missing |

This notebook shows the practical difference.

In [ ]:
import httpx

server = "https://edr.example.com"  # replace with your EDR server root

collections = httpx.get(f"{server}/collections").raise_for_status().json()["collections"]
collection_id = collections[0]["id"]  # or pick any id from the list
collection_url = f"{server}/collections/{collection_id}"

## Mode 1: `probe` (default)

Issues one extra GET to the cube endpoint to retrieve a CoverageJSON,
then reads the actual axis values from `domain.axes`. Result: full
resolution coordinate arrays.

In [ ]:
ds_probe = xr.open_dataset(
    collection_url,
    engine="edr",
    discovery="probe",
)
print("probe mode")
print("  dims:", dict(ds_probe.dims))
print("  x:", ds_probe["x"].values)
print("  y:", ds_probe["y"].values)
print("  t:", ds_probe["t"].values)
ds_probe.close()

## Mode 2: `metadata_only`

Skips the probe. Coordinate arrays come purely from the metadata
extent: spatial bbox corners (`lon_min, lon_max` for `x`; same for
`y`), and `temporal.values` if present (otherwise the interval
endpoints).

This is the fastest mode (no extra request) but lowest resolution. Use
it when you trust the metadata or when the collection is huge and you
only care about coarse navigation.

In [ ]:
ds_meta = xr.open_dataset(
    collection_url,
    engine="edr",
    discovery="metadata_only",
)
print("metadata_only mode")
print("  dims:", dict(ds_meta.dims))
print("  x:", ds_meta["x"].values)  # only 2 corners
print("  y:", ds_meta["y"].values)  # only 2 corners
print("  t:", ds_meta["t"].values)  # all 3 because temporal.values is set
ds_meta.close()

## Mode 3: `strict`

Same axis discovery logic as `metadata_only`, but with a runtime check:
if `extent.temporal.values` is missing from the metadata,
`EdrMetadataError` is raised. Useful when your application requires
deterministic, network-free axis discovery.

In [ ]:
ds_strict = xr.open_dataset(
    collection_url,
    engine="edr",
    discovery="strict",
)
print("strict mode (success — temporal.values present)")
print("  dims:", dict(ds_strict.dims))
ds_strict.close()

### `strict` failure case

If the server's collection metadata does not include explicit `temporal.values`
(only a bbox/interval), `strict` mode raises `EdrMetadataError`:

```python
# This raises EdrMetadataError if the server only advertises an interval,
# not discrete coordinate values.
ds = xr.open_dataset(collection_url, engine="edr", discovery="strict")
```

Use `probe` or `metadata_only` when the server does not expose explicit values.